# STORIES synthetic dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit STORIES and export scores and forecasts

In [ ]:
# OFFICIAL-TUTORIAL STORIES baseline | Synthetic Developmental Crescent v1
# Frozen, public-only; no simulator hidden data, no benchmarking truth, no tuning.
# Author tutorials: https://stories.readthedocs.io/en/latest/vignettes/demo_midbrain.html
#                   https://stories.readthedocs.io/en/latest/vignettes/demo_axolotl.html
# Uses original SpaceTime / fit / compute_potential / compute_velocity / transform APIs.
# Save outputs separate from other baselines. Does not modify any other method.
# Revision: HDF5 StringArray compatibility; safe archiving of this pre-fit crash only.

from pathlib import Path
import os, sys, json, hashlib, platform, time, inspect, warnings
import importlib.metadata as im
import numpy as np
import pandas as pd

PROJECT = Path(os.environ.get("STP_PROJECT", os.environ["STP_PROJECT"]))
DATA = Path(os.environ.get("CRESCENT_DATA_ROOT", str(PROJECT / "Experiments/simulation validation/Synthetic benchmarking/experiments/synthetic_developmental_crescent_v1")))
OUT_BASE = Path(os.environ.get("STORIES_CRESCENT_OUT", str(PROJECT / "experiments/synthetic_benchmark/synthetic_developmental_crescent_v1_stories")))
OUT = OUT_BASE / "tutorial"
PREFLIGHT_ONLY = os.environ.get("STORIES_PREFLIGHT_ONLY", "0") == "1"
GENES = [f"G{g:02d}" for g in range(1, 49)]
SOURCE_STAGE, NEXT_STAGE = 2, 3
SOURCE_T, PRED_T = 2 / 9, 2.5 / 9
TAU = PRED_T - SOURCE_T
PCA_DIM = 20


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def json_save(p, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + ".partial")
    tmp.write_text(json.dumps(obj, sort_keys=True, indent=2, default=str) + "\n")
    os.replace(tmp, p)


def pkg_version(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None


# ---- Strict frozen-data preflight. NEVER search for or read evaluator_only. ----
assert DATA.is_dir(), f"Dataset not mounted at {DATA}"
manifest = json.loads((DATA / "manifests/dataset_manifest.json").read_text())
assert manifest["dataset_id"] == "synthetic_developmental_crescent_v1"
assert manifest["seed"] == 20261023 and manifest["status"] == "frozen"
assert manifest["n_observed_cells"] == 36425
assert manifest["n_source_probes"] == 363
assert manifest["genes"] == GENES
assert np.isclose(float(manifest["source_t"]), SOURCE_T)
assert np.isclose(float(manifest["target_t"]), PRED_T)
public_rel = [
    "train_only/public/observed_cells.csv.gz",
    "train_only/public/counts.csv.gz",
    "train_only/public/genes.csv",
    "manifests/source_probe_cell_ids.csv.gz",
]
input_sha = {}
for rel in public_rel:
    path = DATA / rel
    assert path.is_file(), path
    digest = sha256(path)
    assert digest == manifest["file_sha256"][rel], f"SHA-256 mismatch: {rel}"
    input_sha[rel] = digest

obs = pd.read_csv(DATA / public_rel[0])
count_df = pd.read_csv(DATA / public_rel[1])
gene_df = pd.read_csv(DATA / public_rel[2])
probe_ids = pd.read_csv(DATA / public_rel[3])["cell_id"].astype(str).to_numpy()
assert obs.columns.tolist() == ["cell_id", "tissue_id", "time", "x", "y"]
assert count_df.columns.tolist() == ["cell_id", *GENES]
assert gene_df["gene"].tolist() == GENES
assert len(obs) == len(count_df) == 36425
assert obs.cell_id.is_unique and count_df.cell_id.is_unique
assert np.array_equal(obs.cell_id.to_numpy(dtype=str), count_df.cell_id.to_numpy(dtype=str))
assert np.array_equal(np.sort(obs.tissue_id.unique()), np.arange(10))
assert np.allclose(obs.time, obs.tissue_id / 9)
assert np.allclose(np.sort(obs.time.unique()), np.linspace(0, 1, 10))
assert not np.any(np.isclose(obs.time, PRED_T))
source_mask = obs.tissue_id.to_numpy(dtype=int) == SOURCE_STAGE
source_ids = obs.loc[source_mask, "cell_id"].to_numpy(dtype=str)
assert len(source_ids) == 2419 and len(probe_ids) == len(set(probe_ids)) == 363
assert set(probe_ids).issubset(set(source_ids))
X_raw = count_df[GENES].to_numpy(dtype=np.float32, copy=True)
XY_raw = obs[["x", "y"]].to_numpy(dtype=np.float32, copy=True)
assert np.isfinite(X_raw).all() and (X_raw >= 0).all()
assert np.array_equal(X_raw, np.rint(X_raw))
assert np.isfinite(XY_raw).all()
print("FROZEN PUBLIC DATA VERIFIED", flush=True)
print("  dataset:", DATA)
print("  10 times:", obs.groupby("tissue_id").size().to_dict())
print("  36,425 observed cells; 48 genes; 2,419 source cells; 363 source probes", flush=True)

# Idempotence: verify any prior completed run; never silently replace it.
ALREADY_COMPLETE = (OUT / "complete.json").is_file()
if ALREADY_COMPLETE:
    done = json.loads((OUT / "complete.json").read_text())
    assert done["dataset_id"] == manifest["dataset_id"]
    assert done["input_sha256"] == input_sha
    for rel, digest in done["artifact_sha256"].items():
        path = OUT / rel
        assert path.is_file() and sha256(path) == digest, f"Frozen artifact changed: {rel}"
    print("ALREADY FROZEN: complete.json and all recorded files verified; no retraining.", OUT)
elif OUT.exists() and any(OUT.iterdir()):
    # A previous release of this cell stopped on AnnData's Pandas StringArray
    # HDF5 serialization BEFORE calling SpaceTime.fit(). Preserve its files.
    # Never archive/restart a run that has ANY checkpoint or model output.
    old_files = {p.name for p in OUT.iterdir()}
    prefit_files = {
        "run_config.json", "preprocessing_frozen.npz",
        "public_input_stories_tutorial.h5ad",  # possibly partially written
    }
    recognizes_serialization_failure = (
        {"run_config.json", "preprocessing_frozen.npz"}.issubset(old_files)
        and old_files.issubset(prefit_files)
    )
    if recognizes_serialization_failure:
        try:
            prior_cfg = json.loads((OUT / "run_config.json").read_text())
            recognizes_serialization_failure = (
                prior_cfg.get("dataset_id") == manifest["dataset_id"]
                and prior_cfg.get("input_sha256") == input_sha
            )
        except (ValueError, OSError):
            recognizes_serialization_failure = False
    if PREFLIGHT_ONLY:
        print("PREFLIGHT ONLY: incomplete prior output left untouched:", OUT)
    elif recognizes_serialization_failure:
        archived = OUT_BASE / (
            "tutorial_failed_pretrain_h5ad_"
            + time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
            + f"_{os.getpid()}"
        )
        assert not archived.exists(), f"Refusing to overwrite prior archive: {archived}"
        OUT.rename(archived)  # same-filesystem move; no deletion or replacement
        print("ARCHIVED ONLY THE RECOGNIZED PRE-FIT HDF5 FAILURE:", archived, flush=True)
        print("No fitted parameters/checkpoints were present; restarting the unchanged baseline.", flush=True)
    else:
        raise RuntimeError(
            f"Incomplete or potentially trained STORIES directory exists: {OUT}. "
            "Do not overwrite or retrain. Inspect its files and manually archive if appropriate."
        )

if PREFLIGHT_ONLY:
    print("PREFLIGHT ONLY: public files and manifests passed; no model imported/trained.")

if not PREFLIGHT_ONLY and not ALREADY_COMPLETE:
    import anndata as ad
    import scanpy as sc
    import optax
    import jax
    import stories
    from sklearn.decomposition import PCA
    from flax.serialization import to_bytes
    import matplotlib.pyplot as plt

    print("STORIES installed", pkg_version("stories-jax"), "| JAX", jax.__version__)
    print("JAX devices:", jax.devices(), flush=True)
    if not any(dev.platform == "gpu" for dev in jax.devices()):
        warnings.warn("No GPU detected. The authors recommend GPU acceleration; CPU may be slow.")
    assert hasattr(stories, "SpaceTime") and hasattr(stories.tools, "compute_potential")
    assert hasattr(stories.tools, "compute_velocity")
    assert hasattr(stories.SpaceTime, "transform"), "Installed STORIES lacks native transform()"

    # Public-only, essential adaptation: no precomputed Harmony PCA or batch labels
    # exist in the synthetic panel. Derive unsupervised PCA from all PUBLIC logCP10k
    # observations (not from evaluator-only truth). Retain exactly 20 components.
    adata = ad.AnnData(X=X_raw.copy())
    # Explicit object-backed indices: Pandas 3 may otherwise infer StringArray,
    # which older AnnData cannot serialize to HDF5.
    adata.obs_names = pd.Index(obs.cell_id.to_numpy(dtype=object), dtype=object, name="cell_id")
    adata.var_names = pd.Index(np.asarray(GENES, dtype=object), dtype=object, name="gene")
    adata.obs["time"] = obs.time.to_numpy(dtype=float)  # real clock: 0, 1/9, ... 1
    adata.obs["tissue_id"] = obs.tissue_id.to_numpy(dtype=int)
    adata.layers["counts"] = X_raw.copy()
    adata.obsm["spatial_raw"] = XY_raw.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    log_cp10k = np.asarray(adata.X, dtype=np.float32)
    assert log_cp10k.shape == (36425, 48) and np.isfinite(log_cp10k).all()
    pca = PCA(n_components=PCA_DIM, svd_solver="full")  # deterministic 48-gene PCA
    pca_raw = pca.fit_transform(log_cp10k).astype(np.float32)
    # EXACT tutorial rescaling is divide by the signed/global maximum, not max(abs(.))
    pca_scale = float(np.max(pca_raw))
    assert np.isfinite(pca_scale) and pca_scale > 0, "Unusable PCA scale"
    adata.obsm["X_pca_harmony"] = np.asarray(pca_raw / pca_scale, dtype=np.float32)
    # 'X_pca_harmony' is the tutorial input key, NOT an assertion that Harmony ran.

    # EXACT per-tissue center/axiswise standard deviation as BOTH official tutorials.
    # Spatial input remains raw observed sequencer geometry before this standardization.
    space = XY_raw.astype(np.float64, copy=True)
    spatial_mu = np.zeros((10, 2), dtype=np.float64)
    spatial_std = np.zeros((10, 2), dtype=np.float64)
    for stage in range(10):
        sel = obs.tissue_id.to_numpy(dtype=int) == stage
        spatial_mu[stage] = space[sel].mean(axis=0)
        spatial_std[stage] = space[sel].std(axis=0)
        assert np.all(spatial_std[stage] > 0), f"Degenerate spatial variance: {stage}"
        space[sel] -= spatial_mu[stage]
        space[sel] /= spatial_std[stage]
    adata.obsm["spatial"] = space.astype(np.float32)
    assert np.isfinite(adata.obsm["spatial"]).all()

    # Intentionally OMIT optional 'growth' obs and weight_key in model.fit:
    # STORIES' native DataLoader uses uniform weights when weight_key is None.
    source_code = inspect.getsourcefile(stories.SpaceTime)
    tool_code = inspect.getsourcefile(stories.tools.compute_velocity)
    config = {
        "dataset_id": manifest["dataset_id"], "dataset_seed": 20261023,
        "input_sha256": input_sha,
        "baseline": "STORIES authors' midbrain/axolotl vignette parameters",
        "source_urls": [
            "https://stories.readthedocs.io/en/latest/vignettes/demo_midbrain.html",
            "https://stories.readthedocs.io/en/latest/vignettes/demo_axolotl.html",
        ],
        "implementation": {
            "stories_jax_distribution_version": pkg_version("stories-jax"),
            "stories_space_time_module": str(source_code),
            "stories_space_time_module_sha256": sha256(source_code),
            "stories_tools_module": str(tool_code),
            "stories_tools_module_sha256": sha256(tool_code),
            "jax": jax.__version__, "jaxlib": pkg_version("jaxlib"),
            "flax": pkg_version("flax"), "optax": optax.__version__ if hasattr(optax, "__version__") else pkg_version("optax"),
            "orbax_checkpoint": pkg_version("orbax-checkpoint"),
            "scanpy": pkg_version("scanpy"), "sklearn": pkg_version("scikit-learn"),
            "python": sys.version, "platform": platform.platform(),
            "jax_devices": [str(x) for x in jax.devices()],
        },
        "expression_preprocessing": {
            "raw_input": "48 raw public integer count columns G01-G48",
            "normalization": "scanpy normalize_total(target_sum=1e4), log1p",
            "pca": "sklearn PCA(n_components=20,svd_solver=full), fit on all 10 public snapshots",
            "harmony_applied": False,
            "harmony_note": "No batch labels or author-provided precomputed Harmony PCs available. No fictitious correction.",
            "tutorial_pca_rescaling": "pca_scores /= pca_scores.max()",
            "pca_scale": pca_scale,
            "model_omics_key": "X_pca_harmony (key retained for tutorial API compatibility; ordinary PCA, not Harmony)",
        },
        "space_preprocessing": "observed x,y centered and divided by per-tissue axiswise std; no rigid alignment, no oracle transform",
        "weight_key": None,
        "weights": "native STORIES uniform marginals; no ground-truth population/growth proxy",
        "time_key": "time", "n_timepoints": 10, "timepoints": [i / 9 for i in range(10)],
        "model": {"quadratic_weight": 1e-3, "all_other_model_args": "SpaceTime upstream defaults"},
        "fit": {"optimizer": "optax.adamw(optax.cosine_decay_schedule(1e-2, 10000))",
                "max_iter": 10000, "batch_size": 1000,
                "train_val_split": .75, "patience": 150, "min_delta": 0.,
                "restore": True, "rng_key": "JAX PRNGKey(0)",
                "remaining_fit_args": "upstream defaults"},
        "forecast": {
            "source_stage": SOURCE_STAGE, "target_stage": NEXT_STAGE,
            "source_time": SOURCE_T, "pred_time": PRED_T, "tau": TAU,
            "native_method": "SpaceTime.transform on 2/9 source cells in the trained 20-dimensional PCA representation",
            "gene_adapter": "multiply by saved global PCA scale; sklearn inverse PCA to 48 logCP10k genes; max(log,0); expm1; row normalization",
            "gene_adapter_is_part_of_STORIES": False,
            "notes": "Source-conditioned expression trajectory, NOT lineage assignment or an observed-space migration forecast.",
        },
    }

    # Write input/provenance ONCE. Training outputs are not overwritten on rerun.
    OUT.mkdir(parents=True, exist_ok=False)
    json_save(OUT / "run_config.json", config)
    np.savez_compressed(
        OUT / "preprocessing_frozen.npz", genes=np.asarray(GENES, dtype=str),
        pca_mean=np.asarray(pca.mean_, dtype=np.float64),
        pca_components=np.asarray(pca.components_, dtype=np.float64),
        pca_explained_variance_ratio=np.asarray(pca.explained_variance_ratio_, dtype=np.float64),
        pca_scale=np.asarray(pca_scale, dtype=np.float64),
        space_mu=spatial_mu, space_std=spatial_std,
        timepoints=np.arange(10, dtype=np.float64)/9,
    )
    # HDF5 COMPATIBILITY ONLY (no change to expression, spatial positions or IDs).
    # Important: the error can come from the INDEX called 'cell_id', not only
    # from an obs column. Force both indices and any string-extension columns
    # to ordinary object-backed Python strings before AnnData serialization.
    original_ids = obs.cell_id.to_numpy(dtype=str)
    for df in (adata.obs, adata.var):
        df.index = pd.Index(np.asarray(df.index.tolist(), dtype=object),
                            dtype=object, name=df.index.name)
        for col in df.columns:
            if isinstance(df[col].dtype, pd.StringDtype):
                df[col] = df[col].astype(object)
    assert adata.obs.index.dtype == object and adata.var.index.dtype == object
    assert np.array_equal(adata.obs_names.to_numpy(dtype=str), original_ids)
    assert adata.var_names.to_list() == GENES
    adata.write_h5ad(OUT / "public_input_stories_tutorial.h5ad")
    print("Saved AnnData public input with object-backed string indices.", flush=True)

    # ---- EXACT tutorial model/optimizer/fit call; uses 10k max + default early stopping ----
    model = stories.SpaceTime(quadratic_weight=1e-3)
    scheduler = optax.cosine_decay_schedule(1e-2, 10_000)
    fit_start = time.time()
    model.fit(
        adata=adata,
        time_key="time",
        omics_key="X_pca_harmony",
        space_key="spatial",
        # weight_key is intentionally omitted, using author-defined UNIFORM marginals
        optimizer=optax.adamw(scheduler),
        checkpoint_manager=str((OUT / "ckpt_stories").resolve()),
    )
    fit_seconds = time.time() - fit_start
    print("Official STORIES fit finished in", round(fit_seconds, 2), "seconds; best_step:", model.best_step, flush=True)

    # Preserve restored BEST neural-network parameters as native Flax bytes,
    # in addition to Orbax author's own best-checkpoint tree.
    with open(OUT / "native_stories_best_params.msgpack", "wb") as fh:
        fh.write(to_bytes(model.params))

    def host_float_array(vals):
        return np.asarray([float(np.asarray(jax.device_get(z))) for z in vals], dtype=np.float64)

    train_it = np.asarray(model.train_it, dtype=np.int32)
    val_it = np.asarray(model.val_it, dtype=np.int32)
    train_loss = host_float_array(model.train_losses)
    val_loss = host_float_array(model.val_losses)
    np.savez_compressed(OUT / "native_training_history.npz",
                        train_it=train_it, train_loss=train_loss,
                        val_it=val_it, val_loss=val_loss,
                        best_step=np.asarray(model.best_step))
    assert len(val_it) and np.isfinite(val_loss).all(), "Invalid validation losses: inspect; do not tune"

    # ---- Preserve native author outputs for ALL observed public cells ----
    stories.tools.compute_potential(adata, model, "X_pca_harmony")
    stories.tools.compute_velocity(adata, model, "X_pca_harmony")
    potential = np.asarray(adata.obs["potential"], dtype=np.float32).reshape(-1)
    velocity_pca = np.asarray(adata.obsm["X_velo"], dtype=np.float32)
    assert potential.shape == (36425,) and velocity_pca.shape == (36425, PCA_DIM)
    assert np.isfinite(potential).all() and np.isfinite(velocity_pca).all()
    np.savez_compressed(OUT / "native_observed_potential_velocity.npz",
                        cell_id=obs.cell_id.to_numpy(dtype=str),
                        tissue_id=obs.tissue_id.to_numpy(dtype=np.int32),
                        time=obs.time.to_numpy(dtype=float),
                        potential=potential, velocity_pca=velocity_pca,
                        input_pca=np.asarray(adata.obsm["X_pca_harmony"], dtype=np.float32))

    # ---- NATIVE STORIES prediction of source-cell 20-D latent states at t*=2.5/9 ----
    # Time stays on the original 0..1 scale. Half of the 2/9->3/9 interval is 1/18.
    source_adata = adata[source_mask].copy()
    assert np.array_equal(source_adata.obs_names.to_numpy(dtype=str), source_ids)
    native_latent_pred = np.asarray(
        model.transform(source_adata, omics_key="X_pca_harmony", tau=TAU), dtype=np.float64
    )
    assert native_latent_pred.shape == (2419, PCA_DIM)
    assert np.isfinite(native_latent_pred).all(), "Native forecast nonfinite: numerical failure; no post-hoc tuning"
    np.savez_compressed(OUT / "native_t2_to_t2p5_latent_forecast.npz",
                        source_ids=source_ids, source_probe_ids=np.asarray(probe_ids, dtype=str),
                        genes=np.asarray(GENES, dtype=str),
                        input_pca=source_adata.obsm["X_pca_harmony"].astype(np.float32),
                        pred_pca=native_latent_pred.astype(np.float32),
                        t_source=np.array(SOURCE_T), t_pred=np.array(PRED_T), tau=np.array(TAU))

    # ---- DISCLOSED EVALUATION ADAPTER (NOT A TRAINED STORIES EXPRESSION DECODER) ----
    # Exact inverse affine PCA projection (20 PCs retain less than all 48 dimensions).
    # This is a fixed, public-data-only adapter; it does NOT use future target tissue.
    reconstructed_log = ((native_latent_pred * pca_scale) @ pca.components_.astype(np.float64)
                         + pca.mean_.astype(np.float64))
    assert reconstructed_log.shape == (2419, 48) and np.isfinite(reconstructed_log).all()
    # Mirror the moscot adapter's public-only conversion to nonnegative gene shares.
    nonnegative_log = np.maximum(reconstructed_log, 0.0)
    assert np.max(nonnegative_log) < 700, "Adapter would overflow expm1; report numeric failure, no clipping cap"
    nonnegative_rate = np.expm1(nonnegative_log)
    row_sums = nonnegative_rate.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), "Zero predicted gene-share denominator; report QC failure"
    predicted_shares = (nonnegative_rate / row_sums).astype(np.float32)
    assert np.allclose(predicted_shares.sum(axis=1), 1.0, atol=1e-6)
    np.savez_compressed(OUT / "stories_t2_to_t2p5_frozen_source_forecast.npz",
                        source_ids=source_ids,
                        genes=np.asarray(GENES, dtype=str),
                        source_probe_ids=np.asarray(probe_ids, dtype=str),
                        source_probe_mask=np.isin(source_ids, probe_ids),
                        prediction_logCP10k=reconstructed_log.astype(np.float32),
                        pred_gene_share=predicted_shares,
                        native_prediction_pca=native_latent_pred.astype(np.float32),
                        t_source=np.array(SOURCE_T), t_pred=np.array(PRED_T),
                        tau=np.array(TAU))

    # ---- Read-only public-data diagnostics. All z-up flips are DISPLAY ONLY. ----
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.1), constrained_layout=True)
    ax[0].plot(train_it, train_loss, lw=.85, label="train")
    ax[0].plot(val_it, val_loss, lw=.85, label="val")
    ax[0].axvline(int(model.best_step), color="gray", ls="--", lw=1, label="best checkpoint")
    ax[0].set(xlabel="Iteration", ylabel="STORIES loss", title="Author-default training / validation")
    ax[0].legend(frameon=False)
    ax[1].hist(potential, bins=70)
    ax[1].set(xlabel="Native STORIES potential", ylabel="Observed cells", title="Potential distribution")
    fig.savefig(OUT / "training_and_potential_QC.png", dpi=170)
    plt.show(); plt.close(fig)

    fig, axes = plt.subplots(2, 5, figsize=(16.5, 6.3), constrained_layout=True)
    for stage, ax in enumerate(axes.flat):
        sel = obs.tissue_id.to_numpy(dtype=int) == stage
        scat = ax.scatter(XY_raw[sel, 0], XY_raw[sel, 1], c=potential[sel],
                          s=1.5, cmap="viridis", rasterized=True)
        ax.invert_yaxis()  # high anatomical z UP in display ONLY
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_title(f"t={stage}/9 | n={sel.sum():,}", fontsize=10)
    fig.suptitle("Native STORIES potential on observed crescent (raw spatial positions)")
    fig.colorbar(scat, ax=axes.ravel().tolist(), label="STORIES potential", shrink=.7)
    fig.savefig(OUT / "public_only_stories_potential_QC.png", dpi=170)
    plt.show(); plt.close(fig)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    delta_pc = native_latent_pred - np.asarray(source_adata.obsm["X_pca_harmony"], dtype=np.float64)
    ax[0].hist(np.linalg.norm(delta_pc, axis=1), bins=55)
    ax[0].set(xlabel="||predicted PCA - input PCA||", ylabel="Source cells",
              title="Native source-expression displacement")
    ax[1].scatter(XY_raw[source_mask, 0], XY_raw[source_mask, 1],
                  c=predicted_shares[:, 5], s=2.1, cmap="viridis", rasterized=True)
    ax[1].invert_yaxis(); ax[1].set_aspect("equal", adjustable="datalim")
    ax[1].set_title("Adapter predicted G06 gene share (t*=2.5/9)")
    fig.savefig(OUT / "public_only_source_forecast_QC.png", dpi=170)
    plt.show(); plt.close(fig)

    artifacts = {}
    for fp in sorted(OUT.rglob("*")):
        if fp.is_file() and not fp.name.endswith(".partial") and fp.name != "complete.json":
            artifacts[fp.relative_to(OUT).as_posix()] = sha256(fp)
    json_save(OUT / "complete.json", {
        "dataset_id": manifest["dataset_id"], "input_sha256": input_sha,
        "n_observed": len(obs), "n_genes": len(GENES),
        "n_source": len(source_ids), "n_source_probes": len(probe_ids),
        "fit_seconds": fit_seconds,
        "best_step": int(model.best_step),
        "n_train_iterations": len(train_it), "n_val_iterations": len(val_it),
        "min_recorded_val_loss": float(val_loss.min()),
        "source_pred_pca_shape": list(native_latent_pred.shape),
        "source_pred_gene_shape": list(reconstructed_log.shape),
        "artifact_sha256": artifacts,
        "protocol": "authors' tutorial; ordinary public PCA in place of unavailable precomputed Harmony PCA",
        "adapter_disclosure": "fixed inverse PCA + expm1 gene-share conversion; not STORIES native gene predictor",
    })
    print("\nTRAINING + EXPORT COMPLETE; frozen artifacts:", OUT, flush=True)
    print("  best checkpoint:", model.best_step)
    print("  source latent forecast shape:", native_latent_pred.shape)
    print("  source 48-gene adapter shape:", reconstructed_log.shape)
    print("  native checkpoints and params preserved; no oracle truth opened", flush=True)